In [1]:
import pandas as pd
import duckdb
import os



conn = duckdb.connect('../database/olist.db')

# Orders

In [ ]:
# reading data FROM orders


# Execute and fetch as list of tuples

# Convert the base columns to real timestamps one by one
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE orders ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

result = conn.execute("SELECT * FROM orders").df().head()
print(result)  # [(2,)]


In [ ]:
# calculating number of days till the approval, shipment, delivery

query = """ 
CREATE OR REPLACE TABLE orders_delivery_details AS 
SELECT 
    order_id, customer_id, order_status,
    datediff('day', order_purchase_timestamp, order_approved_at) as days_till_approval,
    datediff('day', order_purchase_timestamp, order_delivered_carrier_date ) as days_till_shipped,
    datediff('day', order_purchase_timestamp , order_delivered_customer_date ) as days_till_delivery,
    datediff('day', order_purchase_timestamp , order_estimated_delivery_date ) as days_till_estim_delivery,
    datediff('day', order_estimated_delivery_date, order_delivered_customer_date) AS delivery_accuracy
FROM orders """
conn.execute(query)


In [ ]:
# reading the order delivery details
query = """
    SELECT * FROM orders_delivery_details
"""
df=conn.execute(query).df()
print(df)

# Products

In [ ]:
# reading data FROM products

# Execute and fetch as list of tuples
result = conn.execute("SELECT * FROM products").df().head()

# Convert the base columns to integers one by one
cols = [
    'product_name_lenght', 
    'product_description_lenght', 
    'product_photos_qty', 
    'product_weight_g', 
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE products ALTER {col} TYPE INTEGER USING TRY_CAST({col} AS INTEGER)")

# Order Items

In [ ]:
# reading data FROM order items

# Convert the base columns to float one by one
cols = [
    'price',
    'freight_value'
]

for col in cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE order_items ALTER {col} TYPE FLOAT USING TRY_CAST({col} AS FLOAT)")

# Convert the base column to date columns
col = 'shipping_limit_date'
print(f"Optimizing {col}...")
conn.execute(f"ALTER TABLE order_items ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

# Execute and fetch as list of tuples
result = conn.execute("SELECT * FROM order_items").df().head()
print(result)


In [ ]:
# aggregating the order items data to calculate total cost, freight value, shipping burden and total cost to customer
query = """
    CREATE OR REPLACE TABLE order_cost_detils AS 
    SELECT 
        order_id, shipping_limit_date, 
        count(distinct product_id) as total_products, 
        count(order_item_id) as total_items, 
        round(SUM(price),4) as total_cost, 
        round(SUM(freight_value),4) as total_freight_value,
        round(SUM(price+freight_value),4) as total_cost_customer,
        round(SUM(freight_value/price),4) as shipping_burden
    FROM order_items
    GROUP BY order_id, shipping_limit_date       
"""
conn.execute(query)

query = """
    SELECT * FROM order_cost_detils
"""
res = conn.execute(query).df().head()
print(res)

In [ ]:
# aggregating order items data to specifically calculate seller related information
query = """
    CREATE OR REPLACE TABLE seller_performance AS 
    SELECT
        seller_id, 
        count(distinct order_id) as total_orders,
        count(distinct product_id) as across_products,
        round(SUM(price),4) as price_value,
        round(SUM(freight_value),4) as freight_value,
        round(SUM(price+freight_value),4) as total_revenue,
        round(SUM(freight_value/price),4) as total_shipping_burden
        FROM order_items
    GROUP BY seller_id

   
"""
conn.execute(query)

query = """
    SELECT * FROM seller_performance
"""
res = conn.execute(query).df().head()
print(res)

# Order Payments

In [ ]:

result = conn.execute("SELECT * FROM order_payments").df().head()
print(result)


                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  


In [ ]:
# aggregating payment information about orders
query = """
    CREATE OR REPLACE TABLE AS order_payments_agg
    SELECT
        order_id, 
            SUM(payment_value) as total_payment_value,
            MAX(payment_installments) as max_pay_install,
            COUNT(DISTINCT payment_type) as type_of_payments,
            SUM(CASE WHEN payment_type = 'credit_card' THEN 1 ELSE 0 END) as credit_card_count,
            SUM(CASE WHEN payment_type = 'debit_card' THEN 1 ELSE 0 END) as debit_card_count,
            SUM(CASE WHEN payment_type = 'voucher' THEN 1 ELSE 0 END) as voucher_count,
            SUM(CASE WHEN payment_type = 'boleto' THEN 1 ELSE 0 END) as boleto_count,
            SUM(CASE WHEN payment_type = 'not_defined' THEN 1 ELSE 0 END) as undef_pay_type
    FROM order_payments
    GROUP BY order_id


"""
result = conn.execute(query).df()
print(result)

                               order_id  total_payment_value  max_pay_install  \
0      2e2c60b99754ae1e4d8b18846cfec9f2               542.66                4   
1      1ffb3c1929b16d9c1aec1958e11b3e9b               166.04                1   
2      3689194c14ad4e2e7361ebd1df0e77b0               153.27                1   
3      a5f860aaf6a3c35399495aefe92d363f                75.75                1   
4      31f8013402a0749b3ab433b1aee11754                42.19                1   
...                                 ...                  ...              ...   
99435  78db0b29049b6795343e2a8ba2f48ff7                84.54                1   
99436  063f2375ecf74934d05e78a476f92cb5                93.50                1   
99437  7b6e1f2c12c784e2ddc7c71547cb6502                23.29                1   
99438  ace1fa00a0f5414248ce3e4c1ccc0c0f                34.05                1   
99439  064558d37369c187c5e1e37eb534ba4d                48.71                1   

       type_of_payments  cr

# Order Reviews

In [48]:

result = conn.execute("SELECT * FROM order_reviews").df()
print(result)


                              review_id                          order_id  \
0      7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1      80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2      228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3      e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4      f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   
...                                 ...                               ...   
99219  574ed12dd733e5fa530cfd4bbf39d7c9  2a8c23fee101d4d5662fa670396eb8da   
99220  f3897127253a9592a73be9bdfdf4ed7a  22ec9f0669f784db00fa86d035cf8602   
99221  b3de70c89b1510c4cd3d0649fd302472  55d4004744368f5571d1f590031933e4   
99222  1adeb9d84d72fe4e337617733eb85149  7725825d039fc1f0ceb7635e3f7d9206   
99223  efe49f1d6f951dd88b51e6ccd4cc548f  90531360ecb1eec2a1fbb265a0db0508   

       review_score review_comment_title  \
0                 4            

In [ ]:
# aggregating reviews data on order basis
query = """
    select 
        order_id, count(review_id) as total_reviews_count,
        round(avg(review_score),2) as avg_score,
        max(review_score) as max_score,
        min(review_score) as min_score,
        count(review_comment_message ) as textual_review
    from order_reviews
    group by order_id
    order by total_reviews_count desc, avg_score desc
"""
result = conn.execute(query).df()
print(result)

                               order_id  total_reviews_count  avg_score  \
0      df56136b8031ecd28e200bb18e6ddb2e                    3       5.00   
1      c88b1d1b157a9999ce368f218a407141                    3       4.33   
2      03c939fd7fd3b38f8485a0f95798f1f6                    3       3.33   
3      8e17072ec97ce29f0e1f111e598b0c85                    3       1.00   
4      2daee070f2042c8b7a8e9fdde778a31a                    2       5.00   
...                                 ...                  ...        ...   
98668  719a2a4825d52ce82cbc8698a71495ba                    1       1.00   
98669  93e87829bab4127b8692f52a73df4284                    1       1.00   
98670  2d31315ad8289e1e63c8451715e80306                    1       1.00   
98671  134f7360b1704296a29b44585e183989                    1       1.00   
98672  9365bcfea1b55fac36333fe520a9a53a                    1       1.00   

       max_score  min_score  textual_review  
0              5          5               0  
1      

# Products Processed

# Sellers

# Customers